In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

## Reading data set

In [2]:
df = pd.read_csv("../Cars_Datasets_2025.csv", encoding="latin-1")

In [3]:
df.columns = df.columns.str.strip()

## CLEANING

In [4]:
df.head()

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


## `Company Names` column text normalization

In [5]:
# Text normalization
df["Company Names"] = df["Company Names"].str.lower().str.strip()

In [6]:
print(df["Company Names"].value_counts().sort_index().to_string())

Company Names
acura                 27
aston martin          11
audi                  21
bentley                1
bmw                   41
bugatti               10
cadillac              20
chevrolet             58
ferrari                9
ford                  54
gmc                   59
honda                  7
hyundai               23
jaguar land rover     40
jeep                  19
kia                   71
lamborghini           24
mahindra               3
maruti suzuki          3
mazda                 87
mercedes              21
mitsubishi            60
nissan               159
peugeot               54
porsche               96
rolls royce           30
tata motors           29
tesla                 10
toyota                50
volkswagen           109
volvo                 12


#### Unique company names without any variation

In [7]:
df.head()

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,ferrari,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,rolls royce,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,mercedes,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,audi,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


# `Car Names` column

In [8]:
df["Cars Names"] = df["Cars Names"].str.strip().str.lower() 

In [9]:
print(len(df["Cars Names"].value_counts().to_string()))

53965


In [10]:
len(df["Cars Names"].value_counts())

1199

### This column has 1199 unique values, almost one per row, so encoding it directly would not help the model (it would only memorize rows). It will be dropped, or used only to extract features.

# `Engines` column

Split engine entries with multiple values into separate rows

In [11]:
df[df["Engines"].str.contains('/')]

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
271,volkswagen,golf,1.5L Turbo I4 / Electric,1498 cc / 40 - 58 kWh,110 - 320 hp,210 km/h,8.5 sec,"$23,000",Petrol,5,250 Nm
275,volkswagen,polo,1.0L Turbo I3 / 1.6L Diesel,999 cc / 1598 cc,80 - 200 hp,190 km/h,9.5 sec,"$18,000",Petrol,5,175 Nm
279,volkswagen,touareg,3.0L V6 Turbo Diesel / 4.0L V8,2967 cc / 3956 cc,231 - 500 hp,235 km/h,5.9 sec,"$55,000","Petrol, Diesel",5,600 Nm
296,volkswagen,t-cross,1.0L Turbo I3 / 1.5L Turbo I4,999 cc / 1498 cc,95 - 150 hp,200 km/h,10.2 sec,"$25,000",Petrol,5,250 Nm
300,volkswagen,multivan,Plug-in Hybrid / Diesel I4,1395 cc / 1968 cc,150 - 218 hp,190 km/h,9.4 sec,"$50,000","Petrol, Hybrid",7,350 Nm
309,volkswagen,golf variant,1.5L Turbo I4 / 2.0L Diesel I4,1498 cc / 1968 cc,130 - 150 hp,220 km/h,7.5 sec,"$32,000",Petrol,5,320 Nm
311,volkswagen,lupo,1.4L I3 Gas / 1.2L Diesel I3,1390 cc / 1199 cc,60 - 75 hp,170 km/h,14.3 sec,"$15,000",Petrol,4,120 Nm
313,volkswagen,phaeton,3.0L V6 Diesel / 6.0L W12 Gas,2967 cc / 5998 cc,240 - 450 hp,250 km/h,6.1 sec,"$85,000",Petrol,5,560 Nm
314,volkswagen,golf cabriolet,1.2L I4 Turbo / 2.0L I4 Turbo,1197 cc / 1984 cc,105 - 210 hp,217 km/h,8.4 sec,"$30,000",Petrol,4,200 Nm
316,volkswagen,polo vivo,1.4L I4 Gas / 1.6L I4 Gas,1390 cc / 1598 cc,85 - 105 hp,170 km/h,10.8 sec,"$14,000",Petrol,5,132 Nm


### Splitting `engines` and `cc/capacity` (explode)

When a cell contains multiple values separated by `/` (e.g. `1.5L Turbo / Electric`), it is split into one row per engine.

- New `_cleaned` columns are created; the original columns are left untouched.
- If a column has a single value, it is repeated to match the length.
- Null values are converted to `[None]`.

In [12]:
df["Engines_cleaned"] = df["Engines"]
df["CC/BatteryCapacity_cleaned"] = df["CC/Battery Capacity"]

cols = ["Engines_cleaned", "CC/BatteryCapacity_cleaned"]
def igualate(row):
    n = max(len(row[c]) for c in cols)
    for c in cols:
        if len(row[c]) == 1:
            row[c] = row[c] * n
    return row

for c in cols:
    df[c] = df[c].str.strip().str.split('/')
    df[c] = df[c].apply(lambda x: x if isinstance(x, list) else [None]) 

df = df.apply(igualate, axis=1)
df = df.explode(cols, ignore_index=True)

In [13]:
df["Engines_cleaned"] = df["Engines_cleaned"].str.strip()

### Creating `displacement` column
Extracting displacement in liters from the engine description using a regex (e.g. `2.0L Turbo I4` → `2.0`) and converting it to a numeric column.

In [14]:
df["displacement"] = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)(\d+\.?\d*)\s*L\b")[0]
)
df["displacement"].isna().sum()

np.int64(480)

In [15]:
df.loc[df["displacement"].isna(), "Engines_cleaned"].value_counts().head(30)

Engines_cleaned
I4                                                64
V12                                               51
V6                                                40
Inline-4                                          38
V8                                                34
Dual Electric Motors                              23
Electric Motor                                    21
I4 Petrol                                         21
Electric                                          16
V6 Petrol                                         13
I3                                                12
V10                                               11
I6                                                 9
V8 Petrol                                          9
Hybrid I4                                          7
I4 + ELECTRIC                                      6
Single Electric Motor                              6
ELECTRIC MOTOR                                     5
Permanent Magnet Synchronous  

Some engines list displacement in cc instead of liters (e.g. `660cc Inline-3`). These values are converted to liters (cc / 1000) and used to fill the missing displacement values.

In [16]:
cc = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)(\d+)\s*cc")[0]
)
df["displacement"] = df["displacement"].fillna(cc / 1000)
df["displacement"].isna().sum()

np.int64(474)

### Creating `cylinders` column

The cylinder count appears in several formats (`V8`, `I4`, `Inline-4`, `Flat-6`, `W16`, `4-Cylinder`...). Each format is extracted with its own regex, and the results are then combined into a single `cylinders` column.

In [17]:
cyl_V = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)V(\d+)")[0]
)

cyl_I = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)I(\d+)")[0]
)

cyl_inline = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)Inline\-*(\d+)")[0]
)

cyl_flat = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)Flat\-*(\d+)")[0]
)

cyl_boxer = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)Boxer\-*(\d+)")[0]
)

cyl_W = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)W\-*(\d+)")[0]
)

cyl_VR = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)VR(\d+)")[0]
)

cyl_cylinder = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)(\d+)\-*Cylinder")[0]
)

df["cylinders"] = (cyl_V.fillna(cyl_I).fillna(cyl_inline).fillna(cyl_flat)
    .fillna(cyl_boxer).fillna(cyl_W).fillna(cyl_VR).fillna(cyl_cylinder).fillna(cyl_cylinder)
)

In [18]:
df["cylinders"].isna().sum()

np.int64(267)

In [19]:
df.loc[df["cylinders"].isna(), "Engines_cleaned"].value_counts().head(20)

Engines_cleaned
Dual Electric Motors                              23
Electric Motor                                    21
Electric                                          16
2.0L MIVEC                                         7
2.4L Diesel Turbo                                  7
Single Electric Motor                              6
ELECTRIC MOTOR                                     5
Permanent Magnet Synchronous                       5
Electric motors with up to 11,500 lb-ft torque     5
Electric (Dual Motor)                              5
1.2L MIVEC                                         5
2.0L SkyActiv-G                                    5
1.4L CR4 Diesel                                    4
2.0L EcoBoost                                      4
3.2L Diesel Turbo                                  4
2.5L SkyActiv-G                                    4
ELECTRIC                                           3
2.8L,TURBO,DIESEL                                  3
1.6L Diesel                   

Electric motors have no cylinders, so their missing values are set to `0`. The `isna()` condition keeps hybrids (e.g. `Hybrid I4 + Electric Motor`) from being overwritten, since they already have a cylinder count.

In [20]:
mask = (
    (df["Engines_cleaned"].str.lower().str.contains("electric") | df["Engines_cleaned"].str.lower().str.contains("permanent magnet")) &
    df["cylinders"].isna()
)

df.loc[mask, "cylinders"] = 0 

In [21]:
df["cylinders"].isna().sum()

np.int64(162)

In [22]:
df.loc[df["cylinders"].isna(), "Engines_cleaned"].value_counts().head(20)

Engines_cleaned
2.0L MIVEC              7
2.4L Diesel Turbo       7
1.2L MIVEC              5
2.0L SkyActiv-G         5
1.4L CR4 Diesel         4
2.0L EcoBoost           4
3.2L Diesel Turbo       4
2.5L SkyActiv-G         4
2.8L,TURBO,DIESEL       3
1.6L Diesel             3
2.0L Diesel             3
1.2L Revotron Petrol    3
1.5L EcoBoost           3
2.0L Hybrid             3
2.0L                    3
2.5L Hybrid             3
EcoBlue Turbo Diesel    3
1.5L MIVEC              3
2.5L Turbocharged       3
1.2L Petrol             2
Name: count, dtype: int64

In [23]:
df.head()

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque,Engines_cleaned,CC/BatteryCapacity_cleaned,displacement,cylinders
0,ferrari,sf90 stradale,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm,V8,3990 cc,NaN,8.0
1,rolls royce,phantom,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm,V12,6749 cc,NaN,12.0
2,ford,ka+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm,1.2L Petrol,"1,200 cc",1.2,NaN
3,mercedes,gt 63 s,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm,V8,"3,982 cc",NaN,8.0
4,audi,audi r8 gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm,V10,"5,204 cc",NaN,10.0


### Creating `is_turbo` column 
Before building the flag, I listed the engines that don't match `turbo`, `supercharged` or `ecoboost` to make sure no turbocharged variant was missed. Then `is_turbo` is created as a binary column (1 = forced induction, 0 = otherwise), and a random sample of flagged rows is reviewed as a sanity check.

In [24]:
text = df["Engines_cleaned"].str.lower()
without_turbo = ~text.str.contains("turbo|supercharged|ecoboost", na=False)
print(text[without_turbo].value_counts().to_string())

Engines_cleaned
i4                                                64
v12                                               51
v6                                                40
inline-4                                          38
v8                                                34
electric motor                                    26
dual electric motors                              23
i4 petrol                                         21
2.0l inline-4                                     20
electric                                          19
1.6l inline-4                                     19
5.6l v8                                           17
5.3l v8 gasoline                                  16
6.2l v8 gasoline                                  15
1.8l inline-4                                     13
v6 petrol                                         13
i3                                                12
3.5l v6                                           12
3.6l v6 gasoline              

In [25]:
df["is_turbo"] = text.str.contains("turbo|supercharged|ecoboost")

In [26]:
df.loc[df["is_turbo"] == 1, "Engines_cleaned"].sample(10)

573        2.9L V6 Twin Turbo
1105    Turbocharged Inline-4
339             2.0L I4 Turbo
327             2.0L I4 Turbo
1006          Turbocharged V6
345             2.0L I4 Turbo
354             1.5L I4 Turbo
394             2.0L I4 Turbo
1070            2.0L EcoBoost
240         2.8L,TURBO,DIESEL
Name: Engines_cleaned, dtype: str

In [27]:
text = text.str.strip()
df["is_hybrid"] = text.str.contains(r"hybrid|phev|plug|\+\s*electric", na=False).astype(int)

In [28]:
print(df.loc[df["is_hybrid"]==0, "Engines_cleaned"].value_counts().to_string())

Engines_cleaned
I4                                                64
V12                                               51
V6                                                40
Inline-4                                          38
V8                                                34
Dual Electric Motors                              23
Electric Motor                                    21
I4 Petrol                                         21
2.0L Inline-4                                     20
1.6L Inline-4                                     19
2.0L I4 Turbo                                     19
5.6L V8                                           17
Electric                                          16
5.3L V8 Gasoline                                  16
3.0L Twin-Turbo Flat-6                            15
6.2L V8 Gasoline                                  15
2.0L Turbo I4                                     14
1.8L Inline-4                                     13
V6 Petrol                     

### Creating `is_diesel` column
Flags diesel engines (1 = diesel, 0 = otherwise). The pattern `diesel?` also catches a typo in the source data (`Turbo Diese`).

In [29]:
df["is_diesel"] = text.str.contains(r"diesel?", na=False).astype(int)

In [30]:
print(df.loc[df["is_diesel"]==0, "Engines_cleaned"].value_counts().to_string())

Engines_cleaned
I4                                                64
V12                                               51
V6                                                40
Inline-4                                          38
V8                                                34
Dual Electric Motors                              23
Electric Motor                                    21
I4 Petrol                                         21
2.0L Inline-4                                     20
1.6L Inline-4                                     19
2.0L I4 Turbo                                     19
5.6L V8                                           17
Electric                                          16
5.3L V8 Gasoline                                  16
3.0L Twin-Turbo Flat-6                            15
6.2L V8 Gasoline                                  15
2.0L Turbo I4                                     14
1.8L Inline-4                                     13
V6 Petrol                     

### Creating `is_electric` column
Flags fully electric powertrains (1 = electric, 0 = otherwise). The pattern covers `electric`, `permanent magnet` and `fuel cell` motors. Hybrids that mention an electric motor (e.g. `I4 + Electric Motor`) are excluded using `is_hybrid`, since they also have a combustion engine.

In [31]:
mask = (
    df["Engines_cleaned"].str.lower().str.strip().str.contains("electric|permanent magnet|fuel cell") &
    (df["is_hybrid"] == 0)
)

df["is_electric"] = mask.astype(int)

In [32]:
df.loc[df["is_electric"] == 0, "Engines_cleaned"]

0                   V8
1                  V12
2          1.2L Petrol
3                   V8
4                  V10
             ...      
1281       2.0L Hybrid
1282              1.8L
1283       2.0L Hybrid
1284       2.5L Hybrid
1285    Plug-in Hybrid
Name: Engines_cleaned, Length: 1183, dtype: str

### Creating `is_rotatory` column
Flags rotary (Wankel) engines (1 = rotary, 0 = otherwise). The pattern covers `rotary`, `wankel` and `rotor`, the different ways these engines are named in the data.

In [33]:
df["is_rotatory"] = text.str.lower().str.strip().str.contains("rotary|wankel|rotor", na=False).astype(int)

In [34]:
print(df.loc[df["is_rotatory"] == 0, "Engines_cleaned"].value_counts().to_string())

Engines_cleaned
I4                                                64
V12                                               51
V6                                                40
Inline-4                                          38
V8                                                34
Dual Electric Motors                              23
Electric Motor                                    21
I4 Petrol                                         21
2.0L Inline-4                                     20
1.6L Inline-4                                     19
2.0L I4 Turbo                                     19
5.6L V8                                           17
Electric                                          16
5.3L V8 Gasoline                                  16
3.0L Twin-Turbo Flat-6                            15
6.2L V8 Gasoline                                  15
2.0L Turbo I4                                     14
1.8L Inline-4                                     13
V6 Petrol                     

### Creating `electric_motors` column
Counts the electric motors of fully electric vehicles from keywords in the description (`single` = 1, `dual` = 2, `tri` = 3, `quad` = 4). Electric engines that don't state a number default to 1, and non-electric engines get 0. Word boundaries (`\b`) prevent `tri` from matching inside `electric`.

In [35]:
number_of_electric_motors = text.str.extract(r"\b(single|dual|tri|quad)\b")[0].map(
    {"single":1, "dual":2, "tri":3, "quad":4}
)

df["electric_motors"] = np.where(df["is_electric"] == 1, number_of_electric_motors.fillna(1), 0).astype(int)

In [36]:
print(df.loc[df["electric_motors"] > 0, "Engines_cleaned"].value_counts().to_string())

Engines_cleaned
Dual Electric Motors                              23
Electric Motor                                    21
Electric                                          16
Single Electric Motor                              6
ELECTRIC MOTOR                                     5
Permanent Magnet Synchronous                       5
Electric motors with up to 11,500 lb-ft torque     5
Electric (Dual Motor)                              5
ELECTRIC                                           3
Tri Electric Motors                                3
Dual Electric Motors (AWD)                         3
Single Electric Motor (RWD)                        2
Dual-Motor Electric                                2
HYDROGEN FUEL CELL,ELECTRIC MOTOR                  1
Electric AWD                                       1
Quad Electric Motors                               1
Electric motors with advanced AWD                  1


In [37]:
df.loc[df["is_electric"] == 1, "displacement"] = 0

In [38]:
df["displacement"].isna().sum()

np.int64(371)

# `CC/BatteryCapacity_cleaned` column

In [39]:
print(df["CC/BatteryCapacity_cleaned"].value_counts().to_string())

CC/BatteryCapacity_cleaned
2,000 cc                                            31
1984 cc                                             31
2,500 cc                                            25
1591 cc                                             22
3996 cc                                             20
1,998 cc                                            19
6200 cc                                             18
3600 cc                                             18
5552 cc                                             17
2894 cc                                             17
1,598 cc                                            16
2981 cc                                             16
1,500 cc                                            16
5300 cc                                             16
6,592 cc                                            15
1998 cc                                             15
2,400 cc                                            15
5,204 cc                              

Checking that rows with only battery capacity (no `cc`) have `is_electric = 1`. The few exceptions are hybrids/PHEVs with a combustion engine, so no correction is needed.

In [40]:
col = df["CC/BatteryCapacity_cleaned"]
with_battery = col.str.contains(r"(?i)kwh|battery|batt", na=False)
with_cc = col.str.contains(r"(?i)\bcc\b", na=False)

rows_with_only_battery = with_battery & ~with_cc

df.loc[rows_with_only_battery, "is_electric"].value_counts()

is_electric
1    100
0     13
Name: count, dtype: int64

All exceptions are hybrids/PHEVs (`is_hybrid = 1`) with a combustion engine, so `is_electric = 0` is correct.

In [41]:
mask = rows_with_only_battery & (df["is_electric"] == 0)
df.loc[mask, ["Engines_cleaned", "CC/BatteryCapacity_cleaned", "is_hybrid", "cylinders", "displacement"]]

,Engines_cleaned,CC/BatteryCapacity_cleaned,is_hybrid,cylinders,displacement
184,I4 + ELECTRIC,13.8 kwh,1,4.0,NaN
192,I4 + ELECTRIC,1.49 kwh,1,4.0,NaN
194,I4 + ELECTRIC,13.8 kwh,1,4.0,NaN
561,2.9L V6 + Electric Motor,17.9 kWh Battery,1,6.0,2.9
562,2.9L V6 + Electric Motor,17.9 kWh Battery,1,6.0,2.9
563,4.0L V8 + Electric Motor,17.9 kWh Battery,1,8.0,4.0
570,4.0L V8 + Electric Motor,17.9 kWh Battery,1,8.0,4.0
711,2.0L Turbo PHEV,17.3 kWh,1,NaN,2.0
712,2.0L Turbo PHEV,17.3 kWh,1,NaN,2.0
713,1.3L Turbo PHEV,11.4 kWh,1,NaN,1.3


### Removing `kwh` rows from `CC/Battery_cleaned` and creating `kwh_capacity`

There are not individual columns with `kwh` and `cc` together 

In [42]:
with_kwh = col.str.contains(r"(?i)kwh", na=False)
(with_kwh & with_cc).sum()

np.int64(0)

Identifying rows that contain `kWh` and creating the `kwh_capacity` column. Rows that do not contain `kWh` are set to 0.

In [43]:
rows_with_kwh = df["CC/BatteryCapacity_cleaned"].str.contains(r"(?i)kwh")
df["kwh_capacity"] = df["CC/BatteryCapacity_cleaned"].where(rows_with_kwh, 0)

Verifying that rows that contains `kwh` in `BatteryCapacity_cleaned` column are the ones with the entire value  

In [44]:
df.loc[df["CC/BatteryCapacity_cleaned"].str.contains(r"(?i)kwh"), "kwh_capacity"].value_counts()

kwh_capacity
77 kWh                                              10
93.4 kWh                                             8
100 kWh                                              8
200 kWh                                              8
87 kWh                                               6
77.4 kWh                                             6
79.2 kWh                                             5
90 kWh (Battery)                                     5
17.9 kWh Battery                                     4
66 kWh                                               4
50 kWh                                               3
95 kwh                                               2
 13.8 kwh                                            2
32.3 kWh                                             2
24 kWh                                               2
82 kWh                                               2
17.3 kWh                                             2
11.4 kWh                                            

In `CC/BatteryCapacity_cleaned` set rows with kwh to `"0"`

In [45]:
df.loc[rows_with_kwh, "CC/BatteryCapacity_cleaned"] = "0" 

### Handle cylinders mistmatch

There is a mismatch in the number of cylinders on row `932`.  
In `CC/BatteryCapacity_cleaned` has `V6(6)`, in `Engines_cleaned` has `V8(8)` and in `cylinders` has `8` 

In [46]:
df.loc[df["CC/BatteryCapacity_cleaned"].str.contains(r"(?i)v"), ["CC/BatteryCapacity_cleaned", "Engines_cleaned", "cylinders"]]

,CC/BatteryCapacity_cleaned,Engines_cleaned,cylinders
241,"HYBRID(2,494 cc),V6(3,456 cc)","2.5L,INLINE-4,HYBRID,OR 3.5L,V6",6.0
931,2995 cc (V6),V6,6.0
932,2995 cc (V6),V8 Petrol,8.0
935,2995 cc (V6),V6 Petrol,6.0
939,2995 cc (V6),V6 Petrol,6.0
942,2995 cc (V6),V6 Petrol,6.0
944,5000 cc (V8),V8 Petrol,8.0
946,5000 cc (V8),V8 Petrol,8.0
949,2995 cc (V6),V6 Petrol,6.0
950,2995 cc (V6),V6 Petrol,6.0


Make sure row `932` is the only one with a mismatch in cylinders.  
Is the only one

In [47]:
batteryCapacity_cleaned_V = df["CC/BatteryCapacity_cleaned"].str.extract(r"(?i)(v\d+)")[0]
engines_cleaned_V = df["Engines_cleaned"].str.extract(r"(?i)(v\d+)")[0]
mismatch = batteryCapacity_cleaned_V.notna() & engines_cleaned_V.notna() & (batteryCapacity_cleaned_V != engines_cleaned_V)

df.loc[mismatch, ["CC/BatteryCapacity_cleaned", "Engines_cleaned", "cylinders"]]

,CC/BatteryCapacity_cleaned,Engines_cleaned,cylinders
932,2995 cc (V6),V8 Petrol,8.0


Looking for the car name of the row `932`.  
It is `jaguar f-type`

In [48]:
df.loc[mismatch, ["Cars Names", "CC/BatteryCapacity_cleaned", "Engines_cleaned", "cylinders"]]

,Cars Names,CC/BatteryCapacity_cleaned,Engines_cleaned,cylinders
932,jaguar f-type,2995 cc (V6),V8 Petrol,8.0


A `jaguar f-type` with `2995cc` is more likely to have `6` cylinders.  
Replace cylinder values in `Engines_cleaned` and `cylinders` columns from `8` to `6`

In [49]:
df.loc[932, "Engines_cleaned"] = "V6 Petrol"
df.loc[932, "cylinders"] = 6.0
df.loc[[932]]

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,...,CC/BatteryCapacity_cleaned,displacement,cylinders,is_turbo,is_hybrid,is_diesel,is_electric,is_rotatory,electric_motors,kwh_capacity
932,jaguar land rover,jaguar f-type,V6 / V8 Petrol,2995 cc (V6),335 hp,260 km/h,5.1 sec,"$70,000",Petrol,2,...,2995 cc (V6),NaN,6.0,False,0,0,0,0,0,0


### Handle `HYBRID(2,494 cc),V6(3,456 cc)` in `CC/BatteryCapacity_cleaned` column

There are a hybrid and gasoline values for the same car

In [50]:
df.loc[df["CC/BatteryCapacity_cleaned"].str.contains("HYBRID(2,494 cc),V6(3,456 cc)", regex=False), ["CC/BatteryCapacity_cleaned", "Engines_cleaned", "Engines"]]

,CC/BatteryCapacity_cleaned,Engines_cleaned,Engines
241,"HYBRID(2,494 cc),V6(3,456 cc)","2.5L,INLINE-4,HYBRID,OR 3.5L,V6","2.5L,INLINE-4,HYBRID,OR 3.5L,V6"


Make a manual `explode`  
Separing the `hybrid` and `gasoline` values into differents rows, and deleting the old row `(241)`

In [51]:
row = df.loc[[241]]
hybrid_row, gasoline_row = row.copy(), row.copy() 

hybrid_row["Engines_cleaned"] = "2.5L Inline-4 Hybrid",
hybrid_row["CC/BatteryCapacity_cleaned"] = "2,494 cc" 
hybrid_row[["displacement", "cylinders", "is_hybrid"]] = [2.5, 4, 1] 

gasoline_row["Engines_cleaned"] = "3.5L V6",
gasoline_row["CC/BatteryCapacity_cleaned"] = "3,456 cc" 
gasoline_row[["displacement", "cylinders", "is_hybrid"]] = [3.5, 6, 0] 

df = pd.concat([df.drop(index=241), hybrid_row, gasoline_row]).reset_index(drop=True)

The `explode` on `Engines` column copied the whole `cc` ranges (e.g., `1,500 - 2,000 cc`) into each resulting row. Each row must keep only its own value.

In [52]:
df.loc[df["CC/BatteryCapacity_cleaned"].str.contains(r"(?=.*cc)(?=.*-)"), ["Engines", "Engines_cleaned", "CC/BatteryCapacity_cleaned"]]

,Engines,Engines_cleaned,CC/BatteryCapacity_cleaned
1055,1.5L EcoBoost / 2.0L Hybrid,1.5L EcoBoost,"1,500 - 2,000 cc"
1056,1.5L EcoBoost / 2.0L Hybrid,2.0L Hybrid,"1,500 - 2,000 cc"
1057,3.5L V6 / 2.0L EcoBoost,3.5L V6,"2,000 - 3,500 cc"
1058,3.5L V6 / 2.0L EcoBoost,2.0L EcoBoost,"2,000 - 3,500 cc"
1059,1.0L EcoBoost / 1.5L,1.0L EcoBoost,"1,000 - 1,500 cc"
1060,1.0L EcoBoost / 1.5L,1.5L,"1,000 - 1,500 cc"
1061,1.0L EcoBoost / 1.5L / 2.0L,1.0L EcoBoost,"1,000 - 2,000 cc"
1062,1.0L EcoBoost / 1.5L / 2.0L,1.5L,"1,000 - 2,000 cc"
1063,1.0L EcoBoost / 1.5L / 2.0L,2.0L,"1,000 - 2,000 cc"
1065,1.0L Turbo Inline-3 / 2.0L,1.0L Turbo Inline-3,"1,000 - 2,000 cc"


Each car already has its engine size in the `displacement` column.  
For rows whose range combines two engines (e.g., `2,000 - 2,200 cc`), take each row's `displacement`, multiply it by `1000` and convert it to `str`.  

Before: `2,000 - 2,200 cc` (in both rows)<br>
After: `2000 cc` (row 1), `2200 cc` (row 2)

In [53]:
mask = df["CC/BatteryCapacity_cleaned"].str.contains(r"(?=.*cc)(?=.*-)")

df.loc[mask, "CC/BatteryCapacity_cleaned"] = (df.loc[mask, "displacement"] * 1000).astype(int).astype(str) + " cc"

In [54]:
df.loc[[1055, 1056]]

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,...,CC/BatteryCapacity_cleaned,displacement,cylinders,is_turbo,is_hybrid,is_diesel,is_electric,is_rotatory,electric_motors,kwh_capacity
1055,ford,fusion (mondeo),1.5L EcoBoost / 2.0L Hybrid,"1,500 - 2,000 cc",160 - 250 hp,220 km/h,7.5 sec,"$25,000 - $40,000",Petrol/Hybrid,5,...,1500 cc,1.5,NaN,True,0,0,0,0,0,0
1056,ford,fusion (mondeo),1.5L EcoBoost / 2.0L Hybrid,"1,500 - 2,000 cc",160 - 250 hp,220 km/h,7.5 sec,"$25,000 - $40,000",Petrol/Hybrid,5,...,2000 cc,2.0,NaN,False,1,0,0,0,0,0


### Removing `(V#)`, `(I#)`, `cc`, `comas (,)` from `CC/BatteryCapacity_cleaned`

Some rows contain `(V#)` or `(I#)` values (e.g., `V6`, `I4`). This information is already in the `Engines_cleaned` and `cylinders` columns, so it is removed from `CC/BatteryCapacity_cleaned`.  
Before: `2,500 cc (V6)`, `3,400 cc (I4)`   
After: `2,500 cc`, `3,400 cc`

In [55]:
df["CC/BatteryCapacity_cleaned"] = df["CC/BatteryCapacity_cleaned"].str.replace(r"(?i)\s*\([vi]\d+\)\s*", "", regex=True).str.strip()
df["CC/BatteryCapacity_cleaned"] = df["CC/BatteryCapacity_cleaned"].str.replace(r"(?i)\s*cc\s*", "", regex = True)
df["CC/BatteryCapacity_cleaned"] = df["CC/BatteryCapacity_cleaned"].str.replace(",", "")

has_number = df["CC/BatteryCapacity_cleaned"].str.contains(r"\d", na=False) 
df.loc[has_number, "CC/BatteryCapacity_cleaned"] = df.loc[has_number, "CC/BatteryCapacity_cleaned"].str.extract(r"\s*(\d+)\s*", expand=False)

In [65]:
df["displacement", ""].head()

0    NaN
1    NaN
2    1.2
3    NaN
4    NaN
Name: displacement, dtype: float64

In [70]:
df[df["Engines_cleaned"] == "V6 EcoBoost"]

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,...,CC/BatteryCapacity_cleaned,displacement,cylinders,is_turbo,is_hybrid,is_diesel,is_electric,is_rotatory,electric_motors,kwh_capacity
1125,ford,bronco (2024),V6 EcoBoost,"2,700 cc",310 hp,200 km/h,6.7 sec,"$45,000",Petrol,5,...,2700,NaN,6.0,True,0,0,0,0,0,0


### In `CC/BatteryCapacity_cleaned` are `0` values where it is suppossed to be a `cc` value

In [57]:
mask = (
    df["CC/BatteryCapacity_cleaned"].str.contains(r"-|\b0\b") & 
    df["CC/Battery Capacity"].str.contains("cc") 
)
df.loc[mask, ["CC/Battery Capacity", "CC/BatteryCapacity_cleaned"]]
#df.loc[df["CC/BatteryCapacity_cleaned"].str.contains(r"-|\b0\b")]

,CC/Battery Capacity,CC/BatteryCapacity_cleaned
184,"1,600 cc / 13.8 kwh",0
192,"1,600 cc / 1.49 kwh",0
194,"1,598 cc / 13.8 kwh",0
275,1498 cc / 40 - 58 kWh,0
1131,"2,400 cc / 20 kWh battery",0


Look in the original column `CC/Battery Capacity` to just take the `cc` value and pass it to `CC/BatteryCapacity_cleaned`

In [58]:
df.loc[mask, "CC/BatteryCapacity_cleaned"] =  (
    df.loc[mask, "CC/Battery Capacity"].str.extract(r"(?i)(\d+,?\d+)\s*cc", expand=False)
)

In [59]:
df.loc[mask, ["CC/Battery Capacity", "CC/BatteryCapacity_cleaned"]]

,CC/Battery Capacity,CC/BatteryCapacity_cleaned
184,"1,600 cc / 13.8 kwh","1,600"
192,"1,600 cc / 1.49 kwh","1,600"
194,"1,598 cc / 13.8 kwh","1,598"
275,1498 cc / 40 - 58 kWh,1498
1131,"2,400 cc / 20 kWh battery","2,400"


### The mayority of rows with `CC/BatteryCapacity_cleaned` in `0` is because they are `electric cars`. But, there are `8 rows` in `Engines_cleaned` which have `displacement` values `(e.g 2.9L)`

In [60]:
mask = (
    (df["CC/BatteryCapacity_cleaned"] == "0") & 
    (df["displacement"] != 0)
)
df.loc[mask, ["CC/BatteryCapacity_cleaned", "Engines_cleaned", "displacement"]]

,CC/BatteryCapacity_cleaned,Engines_cleaned,displacement
560,0,2.9L V6 + Electric Motor,2.9
561,0,2.9L V6 + Electric Motor,2.9
562,0,4.0L V8 + Electric Motor,4.0
569,0,4.0L V8 + Electric Motor,4.0
710,0,2.0L Turbo PHEV,2.0
711,0,2.0L Turbo PHEV,2.0
712,0,1.3L Turbo PHEV,1.3
713,0,1.3L Turbo PHEV,1.3


Let's use it´s `displacement` value to multiply it `1,000` times to get the `cc` value  
Before: `2.9L`  
After: `2,900 (cc value)`

In [61]:
mask = (
    (df["CC/BatteryCapacity_cleaned"] == "0") & 
    (df["displacement"] != 0)
)

df.loc[mask, "CC/BatteryCapacity_cleaned"] = (df.loc[mask, "displacement"] * 1000).astype(str)

In [62]:
df.loc[mask, ["CC/BatteryCapacity_cleaned", "Engines_cleaned", "displacement"]]

,CC/BatteryCapacity_cleaned,Engines_cleaned,displacement
560,2900.0,2.9L V6 + Electric Motor,2.9
561,2900.0,2.9L V6 + Electric Motor,2.9
562,4000.0,4.0L V8 + Electric Motor,4.0
569,4000.0,4.0L V8 + Electric Motor,4.0
710,2000.0,2.0L Turbo PHEV,2.0
711,2000.0,2.0L Turbo PHEV,2.0
712,1300.0,1.3L Turbo PHEV,1.3
713,1300.0,1.3L Turbo PHEV,1.3


### There are `90` values set at `0` in `CC/BatteryCapacity_cleaned` column. Let's verify if its correct

All the cars are electric, so a `0` in `CC/BatteryCapacity_cleaned` it's correct

In [ ]:
print(df.loc[df["CC/BatteryCapacity_cleaned"] == '0', ["CC/BatteryCapacity_cleaned", "Engines_cleaned", "is_electric", "is_hybrid"]].to_string())

In [ ]:
print(df["CC/BatteryCapacity_cleaned"].value_counts().to_string())

In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1287 entries, 0 to 1286
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Company Names               1287 non-null   object 
 1   Cars Names                  1287 non-null   object 
 2   Engines                     1287 non-null   object 
 3   CC/Battery Capacity         1284 non-null   object 
 4   HorsePower                  1287 non-null   object 
 5   Total Speed                 1287 non-null   object 
 6   Performance(0 - 100 )KM/H   1281 non-null   object 
 7   Cars Prices                 1287 non-null   object 
 8   Fuel Types                  1287 non-null   object 
 9   Seats                       1287 non-null   object 
 10  Torque                      1286 non-null   object 
 11  Engines_cleaned             1287 non-null   str    
 12  CC/BatteryCapacity_cleaned  1284 non-null   str    
 13  displacement                916 non-null    